# Unifier Exogenous Feature Analysis

Compare feature correlations across two snapshots:
- **Pre-COVID**: December 2019
- **Latest**: Most recent snapshot

For each data type (ISM, Consumer Confidence, Hours/Wages, Housing, etc.):
1. Side-by-side correlation matrices of all transformations
2. Pairwise correlation tables (sorted by magnitude)
3. Correlation with NFP target (NSA and SA MoM change)

**Unifier series** (12 base): Challenger_Job_Cuts, ISM_Manufacturing_Index, ISM_NonManufacturing_Index,
CB_Consumer_Confidence, AWH_All_Private, AWH_Manufacturing, AHE_Private,
Housing_Starts, Retail_Sales, Empire_State_Mfg, UMich_Expectations, Industrial_Production

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys
import warnings
warnings.filterwarnings('ignore')

# Add project root to path
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from settings import DATA_PATH
from utils.transforms import winsorize_covid_period

# Styling
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 10
pd.set_option('display.max_rows', 200)
pd.set_option('display.max_colwidth', 80)
pd.set_option('display.float_format', lambda x: f'{x:.4f}')

## 1. Load Data

In [ ]:
# --- Paths ---
UNIFIER_DIR = DATA_PATH / "Exogenous_data" / "exogenous_unifier_data" / "decades"
TARGET_DIR = DATA_PATH / "NFP_target"

# --- Load pre-COVID snapshot (Dec 2019) ---
pre_covid_path = UNIFIER_DIR / "2010s" / "2019" / "2019-12.parquet"
snap_pre = pd.read_parquet(pre_covid_path)
snap_pre['date'] = pd.to_datetime(snap_pre['date'])
print(f"Pre-COVID snapshot: {snap_pre['series_name'].nunique()} series, "
      f"{snap_pre['date'].min().date()} to {snap_pre['date'].max().date()}")

# --- Find and load latest snapshot ---
latest_files = sorted(UNIFIER_DIR.rglob('*.parquet'))
latest_path = latest_files[-1]
snap_latest = pd.read_parquet(latest_path)
snap_latest['date'] = pd.to_datetime(snap_latest['date'])
print(f"Latest snapshot: {latest_path.stem} \u2014 {snap_latest['series_name'].nunique()} series, "
      f"{snap_latest['date'].min().date()} to {snap_latest['date'].max().date()}")

# --- Load NFP target data ---
target_nsa = pd.read_parquet(TARGET_DIR / "total_nsa_first_release.parquet")
target_sa  = pd.read_parquet(TARGET_DIR / "total_sa_first_release.parquet")

# Compute MoM difference (these are employment levels, we want changes)
for df in [target_nsa, target_sa]:
    df['ds'] = pd.to_datetime(df['ds'])
    df.sort_values('ds', inplace=True)
    df['y_mom'] = df['y'].diff()

target_nsa = target_nsa.dropna(subset=['y_mom']).set_index('ds')
target_sa  = target_sa.dropna(subset=['y_mom']).set_index('ds')

print(f"\nNSA target: {len(target_nsa)} months ({target_nsa.index.min().date()} to {target_nsa.index.max().date()})")
print(f"SA  target: {len(target_sa)} months ({target_sa.index.min().date()} to {target_sa.index.max().date()})")

In [ ]:
# --- 1b. Drop Short-History Features ---
# Remove any features (series) with fewer than 60 valid (non-NaN) data points
# from BOTH snapshots. These are flagged for manual review.

MIN_VALID_OBS = 60

# Count valid observations per series in each snapshot
valid_counts_latest = snap_latest.groupby('series_name')['value'].count()
valid_counts_pre = snap_pre.groupby('series_name')['value'].count()

# Combined: a feature is "short" if it has < MIN_VALID_OBS in the latest snapshot
short_features = valid_counts_latest[valid_counts_latest < MIN_VALID_OBS].sort_values()

if len(short_features) > 0:
    print(f"\u26a0 FLAGGED: {len(short_features)} features with < {MIN_VALID_OBS} valid data points (dropped from analysis)")
    print(f"  These are typically rare-event indicators or very new series.\n")
    
    # Display flagged features
    flagged_df = pd.DataFrame({
        'Series': short_features.index,
        'Valid_Points_Latest': short_features.values,
        'Valid_Points_PreCOVID': [valid_counts_pre.get(s, 0) for s in short_features.index]
    }).reset_index(drop=True)
    
    display(flagged_df.style.background_gradient(
        subset=['Valid_Points_Latest'], cmap='YlOrRd_r', vmin=0, vmax=MIN_VALID_OBS
    ))
    
    # Remove from both snapshots
    drop_set = set(short_features.index)
    snap_latest = snap_latest[~snap_latest['series_name'].isin(drop_set)]
    snap_pre = snap_pre[~snap_pre['series_name'].isin(drop_set)]
    
    print(f"\n\u2705 Removed {len(drop_set)} short-history features.")
    print(f"   snap_latest: {snap_latest['series_name'].nunique()} series remaining")
    print(f"   snap_pre:    {snap_pre['series_name'].nunique()} series remaining")
else:
    print(f"\u2705 All features have >= {MIN_VALID_OBS} valid data points. No filtering needed.")

## 2. Define Data-Type Groupings

Group the Unifier series by their economic category.

Groups:
- **ISM Surveys**: ISM_Manufacturing_Index, ISM_NonManufacturing_Index
- **Consumer Sentiment**: CB_Consumer_Confidence, UMich_Expectations
- **Hours & Wages**: AWH_All_Private, AWH_Manufacturing, AHE_Private
- **Housing**: Housing_Starts
- **Consumer Demand**: Retail_Sales
- **Industrial Activity**: Industrial_Production
- **Regional PMI**: Empire_State_Mfg
- **Labor Market**: Challenger_Job_Cuts

In [ ]:
# Data type prefixes -- order matters (check longer prefixes first)
DATA_TYPE_PREFIXES = [
    ('ISM_Manufacturing',         'ISM Surveys'),
    ('ISM_NonManufacturing',      'ISM Surveys'),
    ('CB_Consumer_Confidence',    'Consumer Sentiment'),
    ('UMich_Expectations',        'Consumer Sentiment'),
    ('AWH_All_Private',           'Hours & Wages'),
    ('AWH_Manufacturing',         'Hours & Wages'),
    ('AHE_Private',               'Hours & Wages'),
    ('Housing_Starts',            'Housing'),
    ('Retail_Sales',              'Consumer Demand'),
    ('Industrial_Production',     'Industrial Activity'),
    ('Empire_State_Mfg',          'Regional PMI'),
    ('Challenger_Job_Cuts',       'Labor Market'),
]

def classify_series(series_name):
    """Classify a series_name into its data type group."""
    for prefix, group in DATA_TYPE_PREFIXES:
        if series_name.startswith(prefix):
            return group
    return 'Other'

# Get all unique series and classify
all_series = sorted(set(snap_pre['series_name'].unique()) | set(snap_latest['series_name'].unique()))
series_groups = {}
for s in all_series:
    group = classify_series(s)
    series_groups.setdefault(group, []).append(s)

print("Data Type Groups:")
for group, series_list in sorted(series_groups.items()):
    print(f"  {group}: {len(series_list)} series")

## 3. Helper Functions

In [ ]:
def pivot_snapshot(snap_df, series_list):
    """
    Pivot a long-format snapshot to wide format for a specific set of series.
    Takes the LATEST value per date per series (most recent observation).
    """
    mask = snap_df['series_name'].isin(series_list)
    subset = snap_df[mask][['date', 'series_name', 'value']].copy()
    
    # Keep latest observation per (date, series_name)
    subset = subset.drop_duplicates(subset=['date', 'series_name'], keep='last')
    
    wide = subset.pivot(index='date', columns='series_name', values='value')
    wide = wide.sort_index()
    
    # Drop columns/rows that are all NaN
    wide = wide.dropna(axis=1, how='all')
    wide = wide.dropna(axis=0, how='all')
    
    return wide


def shorten_name(name, group_name):
    """
    Shorten series name by removing common prefix for display.
    E.g., 'ISM_Manufacturing_Index_diff_lag_3m' -> 'diff_lag_3m'
    """
    strip_map = {
        'ISM Surveys': ['ISM_Manufacturing_Index_', 'ISM_NonManufacturing_Index_',
                        'ISM_Manufacturing_Index', 'ISM_NonManufacturing_Index'],
        'Consumer Sentiment': ['CB_Consumer_Confidence_', 'UMich_Expectations_',
                               'CB_Consumer_Confidence', 'UMich_Expectations'],
        'Hours & Wages': ['AWH_All_Private_', 'AWH_Manufacturing_', 'AHE_Private_',
                          'AWH_All_Private', 'AWH_Manufacturing', 'AHE_Private'],
        'Housing': ['Housing_Starts_', 'Housing_Starts'],
        'Consumer Demand': ['Retail_Sales_', 'Retail_Sales'],
        'Industrial Activity': ['Industrial_Production_', 'Industrial_Production'],
        'Regional PMI': ['Empire_State_Mfg_', 'Empire_State_Mfg'],
        'Labor Market': ['Challenger_Job_Cuts_', 'Challenger_Job_Cuts'],
    }
    prefixes = strip_map.get(group_name, [])
    for prefix in prefixes:
        if name.startswith(prefix) and len(name) > len(prefix):
            shortened = name[len(prefix):]
            # For multi-series groups, keep the base series identifier
            if group_name in ['ISM Surveys', 'Consumer Sentiment', 'Hours & Wages']:
                # Add short prefix back: ISM_Mfg, ISM_NonMfg, CB_CC, UMich, AWH_Priv, AWH_Mfg, AHE
                base_map = {
                    'ISM_Manufacturing_Index_': 'ISM_Mfg_',
                    'ISM_NonManufacturing_Index_': 'ISM_NMfg_',
                    'CB_Consumer_Confidence_': 'CB_CC_',
                    'UMich_Expectations_': 'UMich_',
                    'AWH_All_Private_': 'AWH_Prv_',
                    'AWH_Manufacturing_': 'AWH_Mfg_',
                    'AHE_Private_': 'AHE_',
                }
                short_prefix = base_map.get(prefix, '')
                return short_prefix + shortened
            return shortened
    return name


def compute_pairwise_correlations(corr_matrix):
    """
    Extract pairwise correlations from a correlation matrix,
    sorted by absolute correlation (descending).
    """
    pairs = []
    cols = corr_matrix.columns
    for i in range(len(cols)):
        for j in range(i+1, len(cols)):
            pairs.append({
                'Feature A': cols[i],
                'Feature B': cols[j],
                'Correlation': corr_matrix.iloc[i, j]
            })
    
    df = pd.DataFrame(pairs)
    df['|Correlation|'] = df['Correlation'].abs()
    df = df.sort_values('|Correlation|', ascending=False).reset_index(drop=True)
    return df


def compute_target_correlations(wide_df, target_series, target_label):
    """
    Compute correlation of each feature with the target MoM change.
    Returns DataFrame sorted by absolute correlation.
    """
    # Align dates
    common_dates = wide_df.index.intersection(target_series.index)
    if len(common_dates) < 10:
        return pd.DataFrame(columns=['Feature', f'Corr with {target_label}', '|Correlation|', 'N'])
    
    target_aligned = target_series.loc[common_dates]
    features_aligned = wide_df.loc[common_dates]
    
    results = []
    for col in features_aligned.columns:
        valid = features_aligned[col].notna() & target_aligned.notna()
        if valid.sum() < 10:
            continue
        corr = features_aligned.loc[valid, col].corr(target_aligned[valid])
        results.append({
            'Feature': col,
            f'Corr with {target_label}': corr,
            '|Correlation|': abs(corr),
            'N': int(valid.sum())
        })
    
    df = pd.DataFrame(results)
    if not df.empty:
        df = df.sort_values('|Correlation|', ascending=False).reset_index(drop=True)
    return df

## 4. Analysis Per Data Type

For each data type, we produce:
1. **Correlation heatmaps** — side-by-side for pre-COVID vs latest
2. **Pairwise correlation tables** — sorted by magnitude
3. **Target correlation tables** — vs NSA MoM and SA MoM for both snapshots

In [ ]:
def analyze_data_type(group_name, series_list):
    """
    Run full analysis for one Unifier data type group.
    """
    print(f"\n{'='*100}")
    print(f"  {group_name}  ({len(series_list)} series)")
    print(f"{'='*100}")
    
    # Pivot to wide format
    wide_pre = pivot_snapshot(snap_pre, series_list)
    wide_latest = pivot_snapshot(snap_latest, series_list)
    
    # Winsorize COVID period (Mar-May 2020) in latest features
    wide_latest = winsorize_covid_period(wide_latest)
    
    if wide_pre.empty or wide_latest.empty:
        print("  \u26a0 Insufficient data for one or both snapshots. Skipping.")
        return
    
    # Shorten column names for display
    short_map_pre = {c: shorten_name(c, group_name) for c in wide_pre.columns}
    short_map_latest = {c: shorten_name(c, group_name) for c in wide_latest.columns}
    wide_pre_short = wide_pre.rename(columns=short_map_pre)
    wide_latest_short = wide_latest.rename(columns=short_map_latest)
    
    # --- PART A: Correlation Matrices (side by side) ---
    print(f"\n--- A. CORRELATION MATRICES ---")
    
    n_features = max(len(wide_pre_short.columns), len(wide_latest_short.columns))
    
    if n_features <= 50:
        corr_pre = wide_pre_short.corr()
        corr_latest = wide_latest_short.corr()
        
        fig, axes = plt.subplots(1, 2, figsize=(min(24, n_features * 0.6 + 6), 
                                                  min(12, n_features * 0.3 + 3)))
        
        sns.heatmap(corr_pre, ax=axes[0], cmap='RdBu_r', center=0, vmin=-1, vmax=1,
                    xticklabels=True, yticklabels=True, square=True,
                    cbar_kws={'shrink': 0.5})
        axes[0].set_title(f'{group_name} \u2014 Pre-COVID (Dec 2019)', fontsize=11, fontweight='bold')
        axes[0].tick_params(axis='both', labelsize=6)
        
        sns.heatmap(corr_latest, ax=axes[1], cmap='RdBu_r', center=0, vmin=-1, vmax=1,
                    xticklabels=True, yticklabels=True, square=True,
                    cbar_kws={'shrink': 0.5})
        axes[1].set_title(f'{group_name} \u2014 Latest ({latest_path.stem}, COVID Winsorized)', fontsize=11, fontweight='bold')
        axes[1].tick_params(axis='both', labelsize=6)
        
        plt.tight_layout()
        plt.show()
    else:
        print(f"  ({n_features} features \u2014 heatmap skipped, see pairwise table below)")
    
    # --- PART B: Pairwise Correlations (sorted) ---
    print(f"\n--- B. PAIRWISE CORRELATIONS (top pairs by |correlation|) ---")
    
    corr_pre_full = wide_pre_short.corr()
    corr_latest_full = wide_latest_short.corr()
    
    pairs_pre = compute_pairwise_correlations(corr_pre_full)
    pairs_latest = compute_pairwise_correlations(corr_latest_full)
    
    print(f"\nPre-COVID top 20 pairs:")
    display(pairs_pre.head(20).style.format({'Correlation': '{:.4f}', '|Correlation|': '{:.4f}'}).background_gradient(
        subset=['|Correlation|'], cmap='YlOrRd', vmin=0, vmax=1))
    
    print(f"\nLatest top 20 pairs:")
    display(pairs_latest.head(20).style.format({'Correlation': '{:.4f}', '|Correlation|': '{:.4f}'}).background_gradient(
        subset=['|Correlation|'], cmap='YlOrRd', vmin=0, vmax=1))
    
    # --- PART C: Target Correlations ---
    print(f"\n--- C. TARGET CORRELATIONS (vs NFP MoM Change) ---")
    
    # Pre-COVID snapshot: correlate with targets up to Dec 2019
    target_nsa_pre = target_nsa.loc[target_nsa.index <= '2019-12-01', 'y_mom']
    target_sa_pre  = target_sa.loc[target_sa.index <= '2019-12-01', 'y_mom']
    
    # Latest snapshot: use all available target data (COVID winsorized)
    target_nsa_all = winsorize_covid_period(target_nsa['y_mom'])
    target_sa_all  = winsorize_covid_period(target_sa['y_mom'])
    
    corr_nsa_pre    = compute_target_correlations(wide_pre, target_nsa_pre, 'NSA MoM')
    corr_sa_pre     = compute_target_correlations(wide_pre, target_sa_pre, 'SA MoM')
    corr_nsa_latest = compute_target_correlations(wide_latest, target_nsa_all, 'NSA MoM')
    corr_sa_latest  = compute_target_correlations(wide_latest, target_sa_all, 'SA MoM')
    
    # Shorten feature names for display
    for df_corr in [corr_nsa_pre, corr_sa_pre, corr_nsa_latest, corr_sa_latest]:
        if not df_corr.empty:
            df_corr['Feature'] = df_corr['Feature'].apply(lambda x: shorten_name(x, group_name))
    
    # Build the unified ranking
    print(f"\nTarget correlation tables (sorted by average |correlation| across all 4):")
    
    all_features = set()
    for df_corr in [corr_nsa_pre, corr_sa_pre, corr_nsa_latest, corr_sa_latest]:
        if not df_corr.empty:
            all_features.update(df_corr['Feature'].tolist())
    
    if not all_features:
        print("  No valid correlations computed.")
        return
    
    merged = pd.DataFrame({'Feature': sorted(all_features)})
    
    for label, df_corr in [
        ('NSA Pre-COVID', corr_nsa_pre),
        ('SA Pre-COVID', corr_sa_pre),
        ('NSA Latest (Winsorized)', corr_nsa_latest),
        ('SA Latest (Winsorized)', corr_sa_latest),
    ]:
        if not df_corr.empty:
            corr_col = [c for c in df_corr.columns if c.startswith('Corr with')][0]
            merge_df = df_corr[['Feature', corr_col]].rename(columns={corr_col: label})
            merged = merged.merge(merge_df, on='Feature', how='left')
    
    # Compute average |correlation| for sorting
    corr_cols = [c for c in merged.columns if c != 'Feature']
    merged['Avg |Corr|'] = merged[corr_cols].abs().mean(axis=1)
    merged = merged.sort_values('Avg |Corr|', ascending=False).reset_index(drop=True)
    
    # Display with color gradient
    display(merged.head(100).style.format(
        {c: '{:.4f}' for c in corr_cols + ['Avg |Corr|']}
    ).background_gradient(
        subset=corr_cols, cmap='RdBu_r', vmin=-0.5, vmax=0.5
    ).background_gradient(
        subset=['Avg |Corr|'], cmap='YlOrRd', vmin=0, vmax=0.5
    ))
    
    return merged

In [ ]:
# Run analysis for all Unifier data types
all_target_results = {}

analysis_order = [
    'ISM Surveys',
    'Consumer Sentiment',
    'Hours & Wages',
    'Housing',
    'Consumer Demand',
    'Industrial Activity',
    'Regional PMI',
    'Labor Market',
]

for group_name in analysis_order:
    if group_name in series_groups:
        result = analyze_data_type(group_name, series_groups[group_name])
        if result is not None:
            all_target_results[group_name] = result

## 5. Cross-Group Summary: Top Features by Target Correlation

In [ ]:
# Combine all target correlation results across groups
all_combined = []
for group_name, df in all_target_results.items():
    df_copy = df.copy()
    df_copy['Data Type'] = group_name
    all_combined.append(df_copy)

if all_combined:
    combined = pd.concat(all_combined, ignore_index=True)
    combined = combined.sort_values('Avg |Corr|', ascending=False).reset_index(drop=True)
    
    print("=" * 100)
    print("  TOP 50 FEATURES BY TARGET CORRELATION (across all Unifier data types)")
    print("=" * 100)
    
    corr_cols = [c for c in combined.columns if c not in ['Feature', 'Data Type', 'Avg |Corr|']]
    
    display(
        combined.head(50)[['Data Type', 'Feature'] + corr_cols + ['Avg |Corr|']].style.format(
            {c: '{:.4f}' for c in corr_cols + ['Avg |Corr|']}
        ).background_gradient(
            subset=corr_cols, cmap='RdBu_r', vmin=-0.5, vmax=0.5
        ).background_gradient(
            subset=['Avg |Corr|'], cmap='YlOrRd', vmin=0, vmax=0.5
        )
    )
else:
    print("No results to combine.")

## 6. Stability Analysis: Pre-COVID vs Latest Correlation Shift

In [ ]:
# For features that appear in both snapshots, compare how their target correlations changed
if all_combined:
    stability = combined.copy()
    
    nsa_pre_col = [c for c in stability.columns if 'NSA Pre' in c]
    nsa_latest_col = [c for c in stability.columns if 'NSA Latest' in c]
    sa_pre_col = [c for c in stability.columns if 'SA Pre' in c]
    sa_latest_col = [c for c in stability.columns if 'SA Latest' in c]
    
    if nsa_pre_col and nsa_latest_col:
        stability['NSA Shift'] = stability[nsa_latest_col[0]] - stability[nsa_pre_col[0]]
    if sa_pre_col and sa_latest_col:
        stability['SA Shift'] = stability[sa_latest_col[0]] - stability[sa_pre_col[0]]
    
    shift_cols = [c for c in ['NSA Shift', 'SA Shift'] if c in stability.columns]
    
    if shift_cols:
        stability['Max |Shift|'] = stability[shift_cols].abs().max(axis=1)
        stability = stability.sort_values('Max |Shift|', ascending=False)
        
        print("=" * 100)
        print("  LARGEST CORRELATION SHIFTS (Pre-COVID \u2192 Latest)")
        print("  Features where correlation with NFP target changed most")
        print("=" * 100)
        
        display_cols = ['Data Type', 'Feature'] + shift_cols + ['Max |Shift|', 'Avg |Corr|']
        display(
            stability.head(30)[display_cols].style.format(
                {c: '{:+.4f}' for c in shift_cols} | {'Max |Shift|': '{:.4f}', 'Avg |Corr|': '{:.4f}'}
            ).background_gradient(
                subset=shift_cols, cmap='RdBu_r', vmin=-0.3, vmax=0.3
            )
        )

In [ ]:
# --- 6. Feature Selection with VIF (Per Group) ---
# Features with < 60 valid points were already removed in cell 1b.
# Now: top 100 correlated -> align to common window -> iterative VIF reduction.
# PRIORITY: Maintain features most correlated with SA WINSORIZED TARGET.

from statsmodels.stats.outliers_influence import variance_inflation_factor
from tqdm.notebook import tqdm

def select_features_vif(X, y, group_name, threshold=10, top_k=100):
    """
    Select features via iterative VIF reduction.
    Prioritizes keeping features with higher correlation to 'y' (SA Target).
    """
    corrs = X.corrwith(y).abs().sort_values(ascending=False)
    
    champion_feature = corrs.index[0] if not corrs.empty else None
    features_above_03 = set(corrs[corrs > 0.3].index)
    has_high_corr = len(features_above_03) > 0
    
    top_features = corrs.head(top_k).index.tolist()
    X_curr = X[top_features].copy()
    
    common_idx = X_curr.dropna().index.intersection(y.dropna().index)
    X_curr = X_curr.loc[common_idx]
    y_curr = y.loc[common_idx]
    
    if len(X_curr) < 20:
        print(f"  Skipping {group_name}: too few rows ({len(X_curr)}) for stable VIF.")
        return []
        
    if len(X_curr) < 100:
        print(f"  Note: {group_name} has {len(X_curr)} common valid rows after alignment.")
    
    iteration = 0
    while X_curr.shape[1] > 1:
        try:
            vifs = [variance_inflation_factor(X_curr.values, i) for i in range(X_curr.shape[1])]
        except Exception as e:
            print(f"  VIF Error: {e}. Stopping.")
            break
        
        max_vif = max(vifs)
        if max_vif <= threshold:
            break
        
        max_idx = vifs.index(max_vif)
        feature_max = X_curr.columns[max_idx]
        
        curr_corr = X_curr.corr().abs()
        collinear_partner = curr_corr[feature_max].drop(feature_max).idxmax()
        
        corr_max = abs(X_curr[feature_max].corr(y_curr))
        corr_partner = abs(X_curr[collinear_partner].corr(y_curr))
        
        if corr_max < corr_partner:
            drop_cand, keep_cand = feature_max, collinear_partner
        else:
            drop_cand, keep_cand = collinear_partner, feature_max
        
        if has_high_corr and drop_cand == champion_feature and corr_max > 0.3:
            drop_cand = keep_cand
        
        if has_high_corr and drop_cand in features_above_03:
            remaining_high = [f for f in X_curr.columns if f in features_above_03 and f != drop_cand]
            if not remaining_high:
                break
        
        X_curr = X_curr.drop(columns=[drop_cand])
        iteration += 1
    
    return X_curr.columns.tolist()


# --- Prepare Data ---
print("Preparing data for VIF Analysis...")
if 'value' in snap_latest.columns:
    df_pivot = snap_latest.pivot(index='date', columns='series_name', values='value').sort_index()
else:
    df_pivot = snap_latest.copy()

# Handle duplicate index entries
df_pivot = df_pivot[~df_pivot.index.duplicated(keep='last')]

# Winsorize COVID period in features
df_pivot = winsorize_covid_period(df_pivot)

# Align with SA Target (Winsorized)
y_sa_all = winsorize_covid_period(target_sa['y_mom'])
y_nsa_all = winsorize_covid_period(target_nsa['y_mom'])

common_dates = df_pivot.index.intersection(y_sa_all.index)
df_pivot = df_pivot.loc[common_dates]
y_target = y_sa_all.loc[common_dates]
y_target_nsa = y_nsa_all.loc[common_dates.intersection(y_nsa_all.index)]

# --- Run Per-Group Selection ---
selected_features_all = {}
final_stats = []

for group_name, series_list in tqdm(sorted(series_groups.items()), desc="VIF Selection"):
    available_series = [s for s in series_list if s in df_pivot.columns]
    
    if len(available_series) < 2:
        continue
    
    X_group = df_pivot[available_series]
    
    try:
        selected_names = select_features_vif(X_group, y_target, group_name)
        if not selected_names:
            continue
            
        selected_features_all[group_name] = selected_names
        
        for feat in selected_names:
            corr_sa = X_group[feat].corr(y_target)
            common_nsa = X_group.index.intersection(y_target_nsa.index)
            corr_nsa = X_group.loc[common_nsa, feat].corr(y_target_nsa.loc[common_nsa]) if not common_nsa.empty else float('nan')
            
            final_stats.append({
                'Group': group_name,
                'Feature': feat,
                'Corr_SA_Winsorized': corr_sa,
                'Corr_NSA_Winsorized': corr_nsa,
                'Abs_Corr_SA': abs(corr_sa)
            })
    except Exception as e:
        print(f"Error processing {group_name}: {e}")

# --- Display Per-Group Results ---
if final_stats:
    df_results = pd.DataFrame(final_stats)
    df_results = df_results.sort_values(['Group', 'Abs_Corr_SA'], ascending=[True, False])
    
    total_selected = len(df_results)
    print(f"\n{'='*80}")
    print(f"  VIF SELECTION RESULTS: {total_selected} features across {len(selected_features_all)} groups")
    print(f"{'='*80}")
    
    for group_name in sorted(df_results['Group'].unique()):
        subset = df_results[df_results['Group'] == group_name][['Feature', 'Corr_SA_Winsorized', 'Corr_NSA_Winsorized']].reset_index(drop=True)
        print(f"\n--- {group_name} ({len(subset)} features) ---")
        display(subset.head(100).style.format({
            'Corr_SA_Winsorized': '{:.4f}',
            'Corr_NSA_Winsorized': '{:.4f}'
        }).background_gradient(cmap='coolwarm', axis=0, subset=['Corr_SA_Winsorized', 'Corr_NSA_Winsorized']))
else:
    print("No features selected.")

In [ ]:
# --- 7. Cross-Group VIF Analysis (Final Feature Set) ---
# Take ALL features selected per-group and run a final VIF reduction.

print("=" * 80)
print("  CROSS-GROUP VIF ANALYSIS (Final)")
print("=" * 80)

if selected_features_all:
    all_selected = []
    feature_to_group = {}
    for group_name, feats in selected_features_all.items():
        all_selected.extend(feats)
        for f in feats:
            feature_to_group[f] = group_name
    
    print("Calculating Global Pairwise Correlations (Winsorized)...")
    global_corrs_sa_signed = df_pivot.corrwith(y_target)
    global_corrs_sa_abs = global_corrs_sa_signed.abs()
    
    global_corrs_nsa_signed = pd.Series(dtype=float)
    if 'y_target_nsa' in locals():
         global_corrs_nsa_signed = df_pivot.corrwith(y_target_nsa)
    
    X_combined = df_pivot[all_selected].copy()
    common_idx = X_combined.dropna().index.intersection(y_target.dropna().index)
    X_clean = X_combined.loc[common_idx]
    y_clean = y_target.loc[common_idx]
    
    print(f"Cross-group matrix: {X_clean.shape[1]} features, {len(X_clean)} rows")
    
    group_champions = {}
    for group_name in selected_features_all.keys():
        group_feats = [f for f in all_selected if feature_to_group.get(f) == group_name]
        if group_feats:
            best_feat = global_corrs_sa_abs[group_feats].idxmax()
            best_corr = global_corrs_sa_abs[best_feat]
            group_champions[group_name] = best_feat
            print(f"  Group '{group_name}' Champion: {best_feat} (Corr={best_corr:.4f})")

    print(f"\n--- Starting VIF Reduction (Threshold=10) ---")
    
    X_final = X_clean.copy()
    iteration = 0
    VIF_THRESHOLD = 10
    dropped_stats = []
    
    while X_final.shape[1] > 1:
        try:
            vifs = pd.Series(
                [variance_inflation_factor(X_final.values, i) for i in range(X_final.shape[1])],
                index=X_final.columns
            )
        except Exception as e:
            print(f"  VIF Error: {e}. Stopping.")
            break
        
        max_vif = vifs.max()
        if max_vif <= VIF_THRESHOLD:
            break
        
        feature_max = vifs.idxmax()
        
        curr_corr = X_final.corr().abs()
        collinear_partner = curr_corr[feature_max].drop(feature_max).idxmax()
        
        corr_max = global_corrs_sa_abs[feature_max]
        corr_partner = global_corrs_sa_abs[collinear_partner]
        
        if corr_max < corr_partner:
            drop_cand, keep_cand = feature_max, collinear_partner
            corr_drop, corr_keep = corr_max, corr_partner
        else:
            drop_cand, keep_cand = collinear_partner, feature_max
            corr_drop, corr_keep = corr_partner, corr_max
            
        drop_group = feature_to_group.get(drop_cand)
        is_champion = (drop_cand == group_champions.get(drop_group))
        
        remaining_in_group = [f for f in X_final.columns if feature_to_group.get(f) == drop_group]
        is_last_survivor = (len(remaining_in_group) == 1)
        
        log_msg = f"  Drop {shorten_name(drop_cand, drop_group)} (VIF={vifs[drop_cand] if drop_cand==feature_max else 'Partner'}, Corr={corr_drop:.4f}) vs {shorten_name(keep_cand, feature_to_group.get(keep_cand, ''))} (Corr={corr_keep:.4f})"
        
        event_type = []
        if is_champion:
            log_msg += " [CHAMPION]"
            event_type.append("Champion")
        if is_last_survivor:
            log_msg += " [LAST SURVIVOR]"
            event_type.append("Last Survivor")
            
        print(log_msg)
        
        if event_type:
            dropped_stats.append({
                'Feature': drop_cand,
                'Group': drop_group,
                'Type': ", ".join(event_type),
                'VIF_at_Drop': vifs[drop_cand] if drop_cand in vifs else vifs.max(),
                'Corr_SA_Winsorized': corr_drop,
                'Corr_NSA_Winsorized': global_corrs_nsa_signed.get(drop_cand, np.nan),
                'Dropped_By': keep_cand
            })
            
        X_final = X_final.drop(columns=[drop_cand])
        iteration += 1
    
    print(f"\nCross-group VIF complete: {X_final.shape[1]} features retained (removed {iteration})")
    
    final_vifs = []
    if not X_final.empty:
        final_vifs = [variance_inflation_factor(X_final.values, i) for i in range(X_final.shape[1])]
    
    final_table = []
    for idx, feat in enumerate(X_final.columns):
        group = feature_to_group.get(feat, 'Unknown')
        
        corr_sa = global_corrs_sa_signed[feat]
        corr_nsa = global_corrs_nsa_signed.get(feat, float('nan'))
        
        final_table.append({
            'Group': group,
            'Feature': feat,
            'Short_Name': shorten_name(feat, group),
            'VIF': final_vifs[idx] if final_vifs else 0,
            'Corr_SA': corr_sa,
            'Corr_NSA': corr_nsa,
            'Abs_Corr_SA': abs(corr_sa)
        })
    
    df_final = pd.DataFrame(final_table).sort_values(['Abs_Corr_SA'], ascending=False).reset_index(drop=True)
    
    print(f"\n--- Final Selected Features ({len(df_final)}) ---")
    display(df_final[['Group', 'Short_Name', 'VIF', 'Corr_SA', 'Corr_NSA']].style.format({
        'VIF': '{:.2f}',
        'Corr_SA': '{:.4f}',
        'Corr_NSA': '{:.4f}'
    }).background_gradient(cmap='coolwarm', axis=0, subset=['Corr_SA', 'Corr_NSA']))
    
    if dropped_stats:
        print(f"\n--- Dropped Champions & Last Survivors ---")
        display(pd.DataFrame(dropped_stats).style.format({
            'VIF_at_Drop': '{:.2f}',
            'Corr_SA_Winsorized': '{:.4f}',
            'Corr_NSA_Winsorized': '{:.4f}'
        }))
else:
    print("No per-group features were selected.")

In [ ]:
# --- 8. Manual Feature Adjustments (Expert Override) ---
# 1. Drop features with NaN correlations.
# 2. Add back specific features, checking VIF.
# 3. CONSISTENCY: Use GLOBAL Pairwise Correlations from previous cell.

print("=" * 80)
print("  MANUAL FEATURE ADJUSTMENTS")
print("=" * 80)

# Check if global correlations exist from previous cell
if 'global_corrs_sa_abs' not in locals():
    print("Computing global correlations (fallback)...")
    global_corrs_sa_signed = df_pivot.corrwith(y_target)
    global_corrs_sa_abs = global_corrs_sa_signed.abs()
    global_corrs_nsa_signed = df_pivot.corrwith(y_target_nsa) if 'y_target_nsa' in locals() else pd.Series()

# Features to DROP (fill in after reviewing VIF results)
features_to_drop = [
    # Add features to drop here after analysis
]

# Features to ADD (fill in after reviewing correlations)
features_to_add = [
    # Add features to add here after analysis
]

# Start with set from Cell 17
if 'df_final' in locals():
    current_features = df_final['Feature'].tolist()
else:
    print("WARNING: `df_final` not found.")
    current_features = []

# Drop
for f in features_to_drop:
    if f in current_features:
        current_features.remove(f)
        print(f"Dropped {f}")

# Add
print("\n--- Adding Features ---")
for f in features_to_add:
    if f not in df_pivot.columns:
        print(f"  WARNING: {f} not in dataset. Skipping.")
        continue
    if f not in current_features:
        current_features.append(f)
        print(f"  Added {f}")
        
    # Check VIF
    X_curr = df_pivot[current_features].copy()
    common = X_curr.dropna().index.intersection(y_target.dropna().index)
    X_clean = X_curr.loc[common]
    
    if X_clean.shape[1] > 0:
        vifs = [variance_inflation_factor(X_clean.values, i) for i in range(X_clean.shape[1])]
        max_vif = max(vifs) if vifs else 0
        print(f"    Max VIF: {max_vif:.2f}")

# Final Table with Global Correlations
print("\n" + "="*80)
print("FINAL TRANSFORMED FEATURE SET (After Manual Adjustments)")
print("="*80)

if current_features:
    X_curr = df_pivot[current_features].copy()
    common = X_curr.dropna().index.intersection(y_target.dropna().index)
    X_clean = X_curr.loc[common]
    final_vifs = [variance_inflation_factor(X_clean.values, i) for i in range(X_clean.shape[1])]
    
    final_rows = []
    for idx, feat in enumerate(X_clean.columns):
        corr_sa = global_corrs_sa_signed.get(feat, np.nan)
        corr_nsa = global_corrs_nsa_signed.get(feat, np.nan)
        
        final_rows.append({
            'Feature': feat,
            'Group': feature_to_group.get(feat, 'Other') if 'feature_to_group' in locals() else 'Other',
            'VIF': final_vifs[idx],
            'Corr_SA': corr_sa,
            'Corr_NSA': corr_nsa,
            'Abs_Corr_SA': abs(corr_sa)
        })
    
    df_final_manual = pd.DataFrame(final_rows).sort_values('Abs_Corr_SA', ascending=False)
    
    display(df_final_manual[['Feature', 'VIF', 'Corr_SA', 'Corr_NSA']].style.format({
        'VIF': '{:.2f}',
        'Corr_SA': '{:.4f}',
        'Corr_NSA': '{:.4f}'
    }).background_gradient(cmap='coolwarm', axis=0, subset=['Corr_SA', 'Corr_NSA']))
    
    print(f"\nFinal Feature Count: {len(df_final_manual)}")
else:
    print("No features in current set.")

In [ ]:
# --- 9. Final Feature Correlation Matrix ---
# Visualize pairwise correlations of the FINAL feature set.

print("=" * 80)
print("  FINAL CORRELATION MATRIX")
print("=" * 80)

final_features = []
if 'df_final_manual' in locals():
    final_features = df_final_manual['Feature'].tolist()
    print(f"Using features from Manual Adjustments ({len(final_features)} features)")
elif 'df_final' in locals():
    final_features = df_final['Feature'].tolist()
    print(f"Using features from VIF Analysis ({len(final_features)} features)")
else:
    print("WARNING: No final feature set found. Please run previous cells.")
    
if final_features:
    X_final = df_pivot[final_features].copy()
    corr_matrix = X_final.corr()
    
    plt.figure(figsize=(24, 20))
    sns.heatmap(
        corr_matrix, 
        annot=False, 
        cmap='coolwarm', 
        center=0,
        vmin=-1, vmax=1,
        square=True,
        linewidths=0.5,
        cbar_kws={"shrink": .5}
    )
    plt.title(f"Pairwise Correlation Matrix: Final {len(final_features)} Unifier Features", fontsize=16)
    plt.tight_layout()
    plt.show()
    
    print("\n--- Remaining High Correlations (>0.7) ---")
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    high_pairs = upper.stack()
    high_pairs_filtered = high_pairs[high_pairs.abs() > 0.7]
    
    if not high_pairs_filtered.empty:
        sorted_pairs = high_pairs_filtered.abs().sort_values(ascending=False)
        for idx, val_abs in sorted_pairs.items():
            val_signed = high_pairs_filtered[idx]
            print(f"{idx[0]} vs {idx[1]}: {val_signed:.4f}")
    else:
        print("None found.")
else:
    print("No features to analyze.")

In [ ]:
# --- 10. NSA Analysis: Within-Group Feature Selection ---
# Replicating the pipeline for NSA Winsorized Target (`y_target_nsa`).

print("=" * 80)
print("  NSA TARGET ANALYSIS: WITHIN-GROUP VIF")
print("=" * 80)

if 'y_target_nsa' not in locals():
    print("Error: `y_target_nsa` not found. Please run Data Loading cells.")
    y_target_nsa_analysis = pd.Series(dtype=float)
else:
    y_target_nsa_analysis = y_target_nsa

def iterative_vif_pruning_nsa(X, y, group_name, top_k=100, threshold=10.0, min_obs=100):
    """
    Iteratively removes features with VIF > threshold.
    Prioritizes keeping features with higher absolute correlation to `y`.
    """
    if X.empty: return []

    common_idx_all = X.index.intersection(y.index)
    X_aligned = X.loc[common_idx_all]
    y_aligned = y.loc[common_idx_all]
    
    valid_counts = X_aligned.notna().sum()
    valid_mask = valid_counts >= min_obs
    X_valid = X_aligned.loc[:, valid_mask]

    if X_valid.empty:
        print(f"  Skipping {group_name}: No features with >= {min_obs} valid observations.")
        return []

    corrs = X_valid.corrwith(y_aligned).abs().sort_values(ascending=False)
    champion_feature = corrs.index[0] if not corrs.empty else None
    features_above_03 = set(corrs[corrs > 0.3].index)
    has_high_corr = len(features_above_03) > 0
    
    top_features = corrs.head(top_k).index.tolist()
    X_curr = X_valid[top_features].copy()
    
    common_idx = X_curr.dropna().index.intersection(y_aligned.dropna().index)
    X_curr = X_curr.loc[common_idx]
    y_curr = y_aligned.loc[common_idx]
    
    if len(X_curr) < 20:
        if champion_feature:
             return [champion_feature]
        return []
    
    iteration = 0
    while X_curr.shape[1] > 1:
        try:
            vifs = [variance_inflation_factor(X_curr.values, i) for i in range(X_curr.shape[1])]
        except Exception as e:
            break
        
        max_vif = max(vifs)
        if max_vif <= threshold:
            break
        
        max_idx = vifs.index(max_vif)
        feature_max = X_curr.columns[max_idx]
        
        curr_corr = X_curr.corr().abs()
        collinear_partner = curr_corr[feature_max].drop(feature_max).idxmax()
        
        corr_max = abs(X_curr[feature_max].corr(y_curr))
        corr_partner = abs(X_curr[collinear_partner].corr(y_curr))
        
        if corr_max < corr_partner:
            drop_cand = feature_max
        else:
            drop_cand = collinear_partner
        
        if has_high_corr and drop_cand == champion_feature and corr_max > 0.3:
            drop_cand = collinear_partner if drop_cand == feature_max else feature_max
            
        X_curr = X_curr.drop(columns=[drop_cand])
        iteration += 1
    
    return X_curr.columns.tolist()

# Run Selection Loop
selected_features_nsa = {}
nsa_stats = []

for group_name in tqdm(sorted(series_groups.keys()), desc="NSA VIF Selection"):
    group_cols = [s for s in series_groups[group_name] if s in df_pivot.columns]
    
    if not group_cols:
        continue
        
    X_group = df_pivot[group_cols]
    
    selected = iterative_vif_pruning_nsa(X_group, y_target_nsa_analysis, group_name)
    selected_features_nsa[group_name] = selected
    
    nsa_stats.append({
        'Group': group_name,
        'Original_Count': len(group_cols),
        'Selected_Count': len(selected)
    })

print("\n--- NSA Selection Summary ---")
display(pd.DataFrame(nsa_stats).set_index('Group'))

In [ ]:
# --- 11. NSA Cross-Group VIF Analysis ---

print("=" * 80)
print("  NSA CROSS-GROUP VIF ANALYSIS")
print("=" * 80)

if selected_features_nsa:
    all_selected_nsa = []
    feature_to_group_nsa = {}
    for group_name, feats in selected_features_nsa.items():
        all_selected_nsa.extend(feats)
        for f in feats:
            feature_to_group_nsa[f] = group_name
            
    print("Calculating Global NSA Correlations...")
    global_corrs_nsa_signed = df_pivot.corrwith(y_target_nsa)
    global_corrs_nsa_abs = global_corrs_nsa_signed.abs()
    
    X_combined = df_pivot[all_selected_nsa].copy()
    common_idx = X_combined.dropna().index.intersection(y_target_nsa.dropna().index)
    X_clean = X_combined.loc[common_idx]
    
    group_champions_nsa = {}
    for group_name in selected_features_nsa.keys():
        group_feats = [f for f in all_selected_nsa if feature_to_group_nsa.get(f) == group_name]
        if group_feats:
            best_feat = global_corrs_nsa_abs[group_feats].idxmax()
            best_corr = global_corrs_nsa_abs[best_feat]
            group_champions_nsa[group_name] = best_feat
            print(f"  Group '{group_name}' Champion: {best_feat} (Corr={best_corr:.4f})")
            
    X_final_nsa = X_clean.copy()
    iteration = 0
    dropped_stats_nsa = []
    
    while X_final_nsa.shape[1] > 1:
        try:
            vifs = pd.Series(
                [variance_inflation_factor(X_final_nsa.values, i) for i in range(X_final_nsa.shape[1])],
                index=X_final_nsa.columns
            )
        except:
            break
            
        max_vif = vifs.max()
        if max_vif <= 10:
            break
            
        feature_max = vifs.idxmax()
        curr_corr = X_final_nsa.corr().abs()
        collinear_partner = curr_corr[feature_max].drop(feature_max).idxmax()
        
        corr_max = global_corrs_nsa_abs[feature_max]
        corr_partner = global_corrs_nsa_abs[collinear_partner]
        
        if corr_max < corr_partner:
            drop_cand, keep_cand = feature_max, collinear_partner
            corr_drop, corr_keep = corr_max, corr_partner
        else:
            drop_cand, keep_cand = collinear_partner, feature_max
            corr_drop, corr_keep = corr_partner, corr_max
            
        drop_group = feature_to_group_nsa.get(drop_cand)
        is_champion = (drop_cand == group_champions_nsa.get(drop_group))
        remaining = [f for f in X_final_nsa.columns if feature_to_group_nsa.get(f) == drop_group]
        is_last = (len(remaining) == 1)
        
        log_msg = f"  Drop {shorten_name(drop_cand, drop_group)} (Corr={corr_drop:.4f}) vs {shorten_name(keep_cand, feature_to_group_nsa.get(keep_cand, ''))} (Corr={corr_keep:.4f})"
        
        event_type = []
        if is_champion:
            log_msg += " [CHAMPION]"
            event_type.append("Champion")
        if is_last:
            log_msg += " [LAST SURVIVOR]"
            event_type.append("Last Survivor")
            
        print(log_msg)
        
        if event_type:
            dropped_stats_nsa.append({
                'Feature': drop_cand,
                'Group': drop_group,
                'Type': ", ".join(event_type),
                'VIF_at_Drop': vifs[drop_cand] if drop_cand in vifs else vifs.max(),
                'Corr_NSA_Winsorized': corr_drop,
                'Dropped_By': keep_cand
            })
            
        X_final_nsa = X_final_nsa.drop(columns=[drop_cand])
        iteration += 1

    print(f"\nNSA Cross-group VIF complete: {X_final_nsa.shape[1]} features retained.")
    
    final_vifs = [variance_inflation_factor(X_final_nsa.values, i) for i in range(X_final_nsa.shape[1])]
    
    nsa_table = []
    for idx, feat in enumerate(X_final_nsa.columns):
        nsa_table.append({
            'Group': feature_to_group_nsa.get(feat, 'Other'),
            'Feature': feat,
            'Short_Name': shorten_name(feat, feature_to_group_nsa.get(feat, 'Other')),
            'VIF': final_vifs[idx],
            'Corr_NSA': global_corrs_nsa_signed[feat],
            'Abs_Corr_NSA': abs(global_corrs_nsa_signed[feat])
        })
        
    df_final_nsa = pd.DataFrame(nsa_table).sort_values('Abs_Corr_NSA', ascending=False)
    
    print(f"\n--- Final Selected Features (NSA Target) ---")
    display(df_final_nsa[['Group', 'Short_Name', 'VIF', 'Corr_NSA']].style.format({
        'VIF': '{:.2f}',
        'Corr_NSA': '{:.4f}'
    }).background_gradient(cmap='coolwarm', axis=0))
    
    if dropped_stats_nsa:
        print(f"\n--- Dropped Champions & Last Survivors (NSA) ---")
        display(pd.DataFrame(dropped_stats_nsa))

In [ ]:
# --- 12. NSA Manual Adjustments ---
# EXPERT OVERRIDE for NSA Feature Set.

print("=" * 80)
print("  NSA MANUAL ADJUSTMENTS")
print("=" * 80)

# Placeholder Lists (fill in after analysis)
features_to_drop_nsa = [
    # Add features to drop here
]

features_to_add_nsa = [
    # Add features to add here
]

current_features_nsa = df_final_nsa['Feature'].tolist() if 'df_final_nsa' in locals() else []

# Drop
for f in features_to_drop_nsa:
    if f in current_features_nsa:
        current_features_nsa.remove(f)
        print(f"Dropped {f}")

# Add
for f in features_to_add_nsa:
    if f not in current_features_nsa and f in df_pivot.columns:
        current_features_nsa.append(f)
        print(f"Added {f}")
        
print(f"Final Count: {len(current_features_nsa)}")

# Recalc Table
if current_features_nsa:
    X_curr = df_pivot[current_features_nsa].copy()
    common = X_curr.dropna().index.intersection(y_target_nsa.dropna().index)
    X_curr = X_curr.loc[common]
    
    final_vifs = [variance_inflation_factor(X_curr.values, i) for i in range(X_curr.shape[1])]
    
    manual_table_nsa = []
    for idx, feat in enumerate(X_curr.columns):
        manual_table_nsa.append({
            'Feature': feat,
            'VIF': final_vifs[idx],
            'Corr_NSA': global_corrs_nsa_signed.get(feat, np.nan)
        })
        
    df_final_manual_nsa = pd.DataFrame(manual_table_nsa).sort_values('Corr_NSA', key=abs, ascending=False)
    display(df_final_manual_nsa)

In [ ]:
# --- 13. NSA Final Correlation Matrix ---
if 'df_final_manual_nsa' in locals() and not df_final_manual_nsa.empty:
    feats = df_final_manual_nsa['Feature'].tolist()
    X_viz = df_pivot[feats].copy()
    corr_matrix = X_viz.corr()
    
    plt.figure(figsize=(24, 20))
    sns.heatmap(corr_matrix, cmap='coolwarm', center=0, vmin=-1, vmax=1, square=True)
    plt.title(f"NSA Target Feature Set: Correlation Matrix ({len(feats)} Unifier features)")
    plt.show()
else:
    print("No features to visualize.")

In [ ]:
# --- 14. Non-Linear Analysis: Mutual Information (MI) ---

print("=" * 80)
print("  MUTUAL INFORMATION SCREENING")
print("=" * 80)

from sklearn.feature_selection import mutual_info_regression
from sklearn.impute import SimpleImputer

def calculate_mi_scores(X, y, top_k=50):
    common = X.index.intersection(y.dropna().index)
    
    if len(common) < 50:
        print(f"  Warning: Only {len(common)} samples for MI analysis.")
        return pd.Series()
        
    X_aligned = X.loc[common].copy()
    y_aligned = y.loc[common].copy()
    
    X_aligned = X_aligned.replace([np.inf, -np.inf], np.nan)
    X_aligned = X_aligned.dropna(axis=1, how='all')
    
    if X_aligned.empty:
        return pd.Series()

    imputer = SimpleImputer(strategy='median')
    X_clean = pd.DataFrame(imputer.fit_transform(X_aligned), columns=X_aligned.columns, index=X_aligned.index)
    
    try:
        mi = mutual_info_regression(X_clean, y_aligned, random_state=42)
        mi = pd.Series(mi, index=X_clean.columns)
        return mi.sort_values(ascending=False).head(top_k)
    except Exception as e:
        print(f"  Error in MI Calc: {e}")
        return pd.Series()

# Filter valid features
valid_cols = [col for col in df_pivot.columns if df_pivot[col].notna().sum() > 100]
print(f"Analyzing {len(valid_cols)} features for Non-Linear Dependency...")
X_all = df_pivot[valid_cols].copy()

# MI for SA
print("\n--- Top 20 MI Scores (SA Target) ---")
mi_sa = calculate_mi_scores(X_all, y_target, top_k=50)
if not mi_sa.empty:
    display(mi_sa.head(20).to_frame(name="MI_Score"))

# MI for NSA
if 'y_target_nsa' in locals():
    print("\n--- Top 20 MI Scores (NSA Target) ---")
    mi_nsa = calculate_mi_scores(X_all, y_target_nsa, top_k=50)
    if not mi_nsa.empty:
        display(mi_nsa.head(20).to_frame(name="MI_Score"))
else:
    mi_nsa = pd.Series()

In [ ]:
# --- 15. Non-Linear Analysis: Random Forest Importance ---

print("=" * 80)
print("  RANDOM FOREST PERMUTATION IMPORTANCE")
print("=" * 80)

from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance

def run_rf_importance(X, y, target_name="Target", top_k=50):
    common = X.index.intersection(y.dropna().index)
    if len(common) < 50:
        return pd.Series()
        
    X_aligned = X.loc[common].copy()
    y_aligned = y.loc[common].copy()
    
    X_aligned = X_aligned.replace([np.inf, -np.inf], np.nan)
    X_aligned = X_aligned.dropna(axis=1, how='all')
    
    imputer = SimpleImputer(strategy='median')
    X_clean = pd.DataFrame(imputer.fit_transform(X_aligned), columns=X_aligned.columns, index=X_aligned.index)
    y_clean = y_aligned
    
    print(f"Training RF on {len(X_clean)} rows, {X_clean.shape[1]} features ({target_name})...")
    rf = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
    rf.fit(X_clean, y_clean)
    print(f"  R2 Score (In-Sample): {rf.score(X_clean, y_clean):.4f}")
    
    print("  Computing Permutation Importance...")
    result = permutation_importance(rf, X_clean, y_clean, n_repeats=5, random_state=42, n_jobs=-1)
    
    importances = pd.Series(result.importances_mean, index=X_clean.columns)
    return importances.sort_values(ascending=False).head(top_k)

# Run for SA
rf_imp_sa = run_rf_importance(X_all, y_target, "SA Target")
print("\n--- Top 20 RF Importance (SA) ---")
if not rf_imp_sa.empty:
    display(rf_imp_sa.head(20).to_frame(name="Importance"))

# Run for NSA
if 'y_target_nsa' in locals():
    rf_imp_nsa = run_rf_importance(X_all, y_target_nsa, "NSA Target")
    print("\n--- Top 20 RF Importance (NSA) ---")
    if not rf_imp_nsa.empty:
        display(rf_imp_nsa.head(20).to_frame(name="Importance"))
else:
    rf_imp_nsa = pd.Series()

In [ ]:
# --- 16. Feature Consolidation (Linear + Non-Linear Candidates) ---

print("=" * 80)
print("  FEATURE CONSOLIDATION")
print("=" * 80)

def consolidate_features(vif_list, mi_series, rf_series, top_n_nonlinear=10):
    final_set = set(vif_list)
    
    mi_top = mi_series.head(top_n_nonlinear).index.tolist()
    rf_top = rf_series.head(top_n_nonlinear).index.tolist()
    
    added_mi = [f for f in mi_top if f not in final_set]
    added_rf = [f for f in rf_top if f not in final_set]
    
    final_set.update(added_mi)
    final_set.update(added_rf)
    
    print(f"Base (VIF): {len(vif_list)}")
    print(f"Added from MI: {len(added_mi)} ({added_mi})")
    print(f"Added from RF: {len(added_rf)} ({added_rf})")
    print(f"Total Unique: {len(final_set)}")
    
    return list(final_set)

# SA Consolidation
print("\n--- SA Target Consolidation ---")
if 'df_final_manual' in locals():
    vif_sa = df_final_manual['Feature'].tolist()
else:
    vif_sa = []

final_sa_candidates = consolidate_features(vif_sa, mi_sa, rf_imp_sa)

# NSA Consolidation
print("\n--- NSA Target Consolidation ---")
if 'df_final_manual_nsa' in locals():
    vif_nsa = df_final_manual_nsa['Feature'].tolist()
else:
    vif_nsa = []

final_nsa_candidates = consolidate_features(vif_nsa, mi_nsa, rf_imp_nsa)

In [ ]:
# --- 17. Final Consolidated Matrix ---

# SA Plot
if final_sa_candidates:
    plt.figure(figsize=(24, 20))
    sns.heatmap(df_pivot[final_sa_candidates].corr(), cmap='coolwarm', center=0, vmin=-1, vmax=1)
    plt.title(f"Final Consolidated SA Unifier Features ({len(final_sa_candidates)})")
    plt.show()

# NSA Plot
if final_nsa_candidates:
    plt.figure(figsize=(24, 20))
    sns.heatmap(df_pivot[final_nsa_candidates].corr(), cmap='coolwarm', center=0, vmin=-1, vmax=1)
    plt.title(f"Final Consolidated NSA Unifier Features ({len(final_nsa_candidates)})")
    plt.show()